# Capítulo 4. Aplicación del Método de Monte Carlo a un modelo balístico

Este cuaderno acompaña al capítulo dedicado a la aplicación balística del método de Monte Carlo. Se estudia el movimiento de un proyectil en un plano vertical bajo la acción de la gravedad y una fuerza de rozamiento lineal proporcional a la velocidad.

El objetivo del notebook es doble:

1. resolver numéricamente el modelo balístico determinista;
2. introducir incertidumbre en los parámetros y estimar mediante Monte Carlo el alcance medio, la dispersión, intervalos de confianza y probabilidades de impacto.

Todas las variables principales del código se escriben en castellano para facilitar su lectura en el contexto del TFG.

## 1. Librerías y configuración general

Se emplean `numpy` para el cálculo numérico, `matplotlib` para las gráficas y `scipy.optimize` para resolver el tiempo de vuelo. En caso de no disponer de `scipy`, se proporciona también una función elemental de bisección.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.optimize import brentq
    SCIPY_DISPONIBLE = True
except Exception:
    SCIPY_DISPONIBLE = False

# Para reproducibilidad de los experimentos Monte Carlo
semilla = 12345
generador = np.random.default_rng(semilla)

# Constante gravitatoria terrestre en m/s^2
gravedad = 9.81

print("SciPy disponible:", SCIPY_DISPONIBLE)

## 2. Modelo balístico con rozamiento lineal

Consideramos una partícula de masa $m>0$, sometida a la gravedad y a una fuerza de rozamiento lineal

$$
\mathbf F_R=-c\mathbf v.
$$

Si $\lambda=c/m$, las ecuaciones del movimiento son

$$
\ddot x=-\lambda \dot x,\qquad
\ddot y=-\lambda \dot y-g.
$$

Con condiciones iniciales

$$
x(0)=0,\quad y(0)=0,\quad
\dot x(0)=v_0\cos\theta,\quad
\dot y(0)=v_0\sin\theta,
$$

la solución explícita, para $\lambda>0$, es

$$
x(t)=\frac{v_0\cos\theta}{\lambda}\left(1-e^{-\lambda t}\right),
$$

$$
y(t)=\frac{v_0\sin\theta+g/\lambda}{\lambda}\left(1-e^{-\lambda t}\right)-\frac{g}{\lambda}t.
$$

Cuando $c=0$, se recupera el tiro parabólico clásico.

In [ ]:
def posicion_balistico(tiempo, velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad=9.81):
    """
    Calcula la posición (x(t), y(t)) de un proyectil con rozamiento lineal.

    Parámetros
    ----------
    tiempo : float o array
        Tiempo o vector de tiempos.
    velocidad_inicial : float
        Velocidad inicial v0, en m/s.
    angulo_lanzamiento : float
        Ángulo de lanzamiento en radianes.
    coeficiente_rozamiento : float
        Coeficiente de rozamiento lineal c.
    masa : float
        Masa del proyectil m, en kg.
    gravedad : float
        Aceleración gravitatoria, en m/s^2.

    Devuelve
    --------
    x, y : arrays o floats
        Coordenadas horizontal y vertical.
    """
    tiempo = np.asarray(tiempo, dtype=float)
    velocidad_x0 = velocidad_inicial * np.cos(angulo_lanzamiento)
    velocidad_y0 = velocidad_inicial * np.sin(angulo_lanzamiento)

    if abs(coeficiente_rozamiento) < 1e-14:
        x = velocidad_x0 * tiempo
        y = velocidad_y0 * tiempo - 0.5 * gravedad * tiempo**2
        return x, y

    lambda_rozamiento = coeficiente_rozamiento / masa
    factor = 1.0 - np.exp(-lambda_rozamiento * tiempo)

    x = (velocidad_x0 / lambda_rozamiento) * factor
    y = ((velocidad_y0 + gravedad / lambda_rozamiento) / lambda_rozamiento) * factor - (gravedad / lambda_rozamiento) * tiempo
    return x, y


def altura_balistico(tiempo, velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad=9.81):
    """Devuelve únicamente la altura y(t)."""
    _, y = posicion_balistico(tiempo, velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad)
    return y

## 3. Tiempo de vuelo y alcance

El tiempo de vuelo $T_f$ se define como la primera raíz positiva de

$$
y(T_f)=0.
$$

Una vez calculado $T_f$, el alcance horizontal es

$$
R=x(T_f).
$$

La siguiente función calcula ambos valores. Se usa un método robusto de búsqueda de raíces. Primero se construye un intervalo donde la altura cambia de signo y después se aplica `brentq` o, si no está disponible, una bisección elemental.

In [ ]:
def biseccion(funcion, extremo_izquierdo, extremo_derecho, tolerancia=1e-10, max_iteraciones=200):
    """Método de bisección elemental para una raíz con cambio de signo."""
    a = extremo_izquierdo
    b = extremo_derecho
    fa = funcion(a)
    fb = funcion(b)

    if fa * fb > 0:
        raise ValueError("La función no cambia de signo en el intervalo dado.")

    for _ in range(max_iteraciones):
        m = 0.5 * (a + b)
        fm = funcion(m)
        if abs(fm) < tolerancia or 0.5 * (b - a) < tolerancia:
            return m
        if fa * fm <= 0:
            b = m
            fb = fm
        else:
            a = m
            fa = fm
    return 0.5 * (a + b)


def tiempo_vuelo(velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad=9.81):
    """
    Calcula el tiempo de vuelo como la primera raíz positiva de y(t)=0.
    """
    if velocidad_inicial <= 0 or masa <= 0:
        return np.nan

    # Caso sin rozamiento: solución explícita.
    if abs(coeficiente_rozamiento) < 1e-14:
        return max(0.0, 2.0 * velocidad_inicial * np.sin(angulo_lanzamiento) / gravedad)

    def funcion_altura(t):
        return float(altura_balistico(t, velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad))

    # Evitamos la raíz trivial t=0 empezando en un tiempo pequeño.
    extremo_izquierdo = 1e-8

    # Cota inicial inspirada en el tiempo de vuelo sin rozamiento, ampliada.
    tiempo_sin_rozamiento = max(1e-6, 2.0 * velocidad_inicial * np.sin(angulo_lanzamiento) / gravedad)
    extremo_derecho = max(1.0, 2.0 * tiempo_sin_rozamiento)

    # Ampliamos el intervalo hasta que y(t)<0.
    while funcion_altura(extremo_derecho) > 0:
        extremo_derecho *= 2.0
        if extremo_derecho > 1e5:
            return np.nan

    if SCIPY_DISPONIBLE:
        return brentq(funcion_altura, extremo_izquierdo, extremo_derecho)
    else:
        return biseccion(funcion_altura, extremo_izquierdo, extremo_derecho)


def alcance_balistico(velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad=9.81):
    """Calcula el alcance horizontal del proyectil."""
    tf = tiempo_vuelo(velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad)
    if not np.isfinite(tf):
        return np.nan
    x_final, _ = posicion_balistico(tf, velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa, gravedad)
    return float(x_final)

## 4. Trayectoria nominal

Fijamos unos valores nominales de los parámetros y representamos la trayectoria. Estos valores pueden modificarse según el caso didáctico deseado.

In [ ]:
# Parámetros nominales
velocidad_inicial_nominal = 300.0          # m/s
angulo_lanzamiento_nominal_grados = 45.0  # grados
angulo_lanzamiento_nominal = np.deg2rad(angulo_lanzamiento_nominal_grados)
coeficiente_rozamiento_nominal = 0.15     # kg/s, en un modelo lineal simplificado
masa_nominal = 10.0                       # kg

# Tiempo de vuelo y alcance nominales
tiempo_vuelo_nominal = tiempo_vuelo(
    velocidad_inicial_nominal,
    angulo_lanzamiento_nominal,
    coeficiente_rozamiento_nominal,
    masa_nominal,
    gravedad
)

alcance_nominal = alcance_balistico(
    velocidad_inicial_nominal,
    angulo_lanzamiento_nominal,
    coeficiente_rozamiento_nominal,
    masa_nominal,
    gravedad
)

print(f"Tiempo de vuelo nominal: {tiempo_vuelo_nominal:.3f} s")
print(f"Alcance nominal: {alcance_nominal:.3f} m")

In [ ]:
tiempos = np.linspace(0, tiempo_vuelo_nominal, 400)
x_nominal, y_nominal = posicion_balistico(
    tiempos,
    velocidad_inicial_nominal,
    angulo_lanzamiento_nominal,
    coeficiente_rozamiento_nominal,
    masa_nominal,
    gravedad
)

plt.figure(figsize=(9, 5))
plt.plot(x_nominal, y_nominal, label="Trayectoria nominal")
plt.axhline(0, linewidth=1)
plt.xlabel("Distancia horizontal x (m)")
plt.ylabel("Altura y (m)")
plt.title("Trayectoria balística con rozamiento lineal")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Incertidumbre en los parámetros

En una situación real, los parámetros no suelen conocerse exactamente. Modelamos el vector aleatorio

$$
\Theta=(V_0,\Theta_0,C,M),
$$

donde:

- $V_0$ es la velocidad inicial;
- $\Theta_0$ es el ángulo de lanzamiento;
- $C$ es el coeficiente de rozamiento lineal;
- $M$ es la masa del proyectil.

En este notebook se usan distribuciones normales truncadas o recortadas de manera sencilla para evitar valores físicamente imposibles. Esta elección es didáctica; en una aplicación real habría que ajustar las distribuciones a datos experimentales.

In [ ]:
def generar_parametros_aleatorios(numero_simulaciones, generador):
    """
    Genera muestras aleatorias de los parámetros balísticos.

    Todas las variables se devuelven en unidades SI y el ángulo en radianes.
    """
    velocidad_inicial = generador.normal(loc=300.0, scale=10.0, size=numero_simulaciones)
    angulo_lanzamiento_grados = generador.normal(loc=45.0, scale=2.0, size=numero_simulaciones)
    coeficiente_rozamiento = generador.normal(loc=0.15, scale=0.02, size=numero_simulaciones)
    masa = generador.normal(loc=10.0, scale=0.5, size=numero_simulaciones)

    # Recortes físicos simples
    velocidad_inicial = np.clip(velocidad_inicial, 1.0, None)
    angulo_lanzamiento_grados = np.clip(angulo_lanzamiento_grados, 1.0, 89.0)
    coeficiente_rozamiento = np.clip(coeficiente_rozamiento, 1e-6, None)
    masa = np.clip(masa, 0.1, None)

    angulo_lanzamiento = np.deg2rad(angulo_lanzamiento_grados)

    return velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa

## 6. Simulación Monte Carlo del alcance

Para cada realización aleatoria de los parámetros, resolvemos el problema balístico y obtenemos un alcance. Así se genera una muestra

$$
R_1,R_2,\ldots,R_N.
$$

El estimador Monte Carlo del alcance medio es

$$
\widehat \mu_R=\frac{1}{N}\sum_{i=1}^N R_i.
$$

La desviación típica muestral permite construir un intervalo de confianza aproximado del 95%:

$$
\widehat\mu_R\pm 1.96\frac{s_R}{\sqrt{N}}.
$$

In [ ]:
def simular_alcances(numero_simulaciones, generador, gravedad=9.81):
    """Simula alcances balísticos mediante Monte Carlo."""
    velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa = generar_parametros_aleatorios(
        numero_simulaciones,
        generador
    )

    alcances = np.empty(numero_simulaciones)

    for indice in range(numero_simulaciones):
        alcances[indice] = alcance_balistico(
            velocidad_inicial[indice],
            angulo_lanzamiento[indice],
            coeficiente_rozamiento[indice],
            masa[indice],
            gravedad
        )

    return alcances, velocidad_inicial, angulo_lanzamiento, coeficiente_rozamiento, masa


def resumen_montecarlo(valores, nivel_confianza=0.95):
    """Devuelve media, desviación típica y semianchura de IC del 95% aproximadamente."""
    valores = np.asarray(valores, dtype=float)
    valores = valores[np.isfinite(valores)]
    numero = len(valores)
    media = np.mean(valores)
    desviacion_tipica = np.std(valores, ddof=1)

    # Para 95%; se deja explícito para mantener el cuaderno elemental.
    z = 1.96
    error_tipico = desviacion_tipica / np.sqrt(numero)
    semianchura = z * error_tipico

    return {
        "numero": numero,
        "media": media,
        "desviacion_tipica": desviacion_tipica,
        "error_tipico": error_tipico,
        "semianchura_95": semianchura,
        "limite_inferior_95": media - semianchura,
        "limite_superior_95": media + semianchura,
    }

In [ ]:
numero_simulaciones = 5000

alcances, velocidades, angulos, coeficientes, masas = simular_alcances(
    numero_simulaciones,
    generador,
    gravedad
)

resumen = resumen_montecarlo(alcances)

for clave, valor in resumen.items():
    if clave == "numero":
        print(f"{clave}: {valor}")
    else:
        print(f"{clave}: {valor:.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(alcances, bins=40, density=True, alpha=0.75)
plt.axvline(resumen["media"], linestyle="--", linewidth=2, label="Media Monte Carlo")
plt.axvline(alcance_nominal, linestyle=":", linewidth=2, label="Alcance nominal")
plt.xlabel("Alcance R (m)")
plt.ylabel("Densidad estimada")
plt.title("Distribución empírica del alcance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Visualización de trayectorias aleatorias

Además de estudiar el alcance, es útil representar algunas trayectorias generadas con parámetros aleatorios. Esto permite visualizar la dispersión balística del modelo.

In [ ]:
numero_trayectorias = 40
indices = generador.choice(len(alcances), size=numero_trayectorias, replace=False)

plt.figure(figsize=(9, 5))

for indice in indices:
    tf = tiempo_vuelo(
        velocidades[indice],
        angulos[indice],
        coeficientes[indice],
        masas[indice],
        gravedad
    )
    tiempos_i = np.linspace(0, tf, 150)
    x_i, y_i = posicion_balistico(
        tiempos_i,
        velocidades[indice],
        angulos[indice],
        coeficientes[indice],
        masas[indice],
        gravedad
    )
    plt.plot(x_i, y_i, alpha=0.35)

plt.plot(x_nominal, y_nominal, linewidth=3, label="Trayectoria nominal")
plt.axhline(0, linewidth=1)
plt.xlabel("Distancia horizontal x (m)")
plt.ylabel("Altura y (m)")
plt.title("Familia de trayectorias bajo incertidumbre")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Probabilidad de impacto

Sea $R_{\rm obj}$ una distancia objetivo y sea $\Delta>0$ una tolerancia. Definimos el suceso de impacto como

$$
|R-R_{\rm obj}|\leq \Delta.
$$

La probabilidad de impacto es

$$
p=\mathbb P\left(|R-R_{\rm obj}|\leq \Delta\right).
$$

Su estimador Monte Carlo es

$$
\widehat p_N=\frac{1}{N}\sum_{i=1}^N \mathbf 1_{\{|R_i-R_{\rm obj}|\leq \Delta\}}.
$$

In [ ]:
objetivo = alcance_nominal      # distancia objetivo en metros
tolerancia = 100.0              # tolerancia admisible en metros

impactos = np.abs(alcances - objetivo) <= tolerancia
probabilidad_impacto = np.mean(impactos)

# Error típico aproximado para una proporción
error_tipico_impacto = np.sqrt(probabilidad_impacto * (1.0 - probabilidad_impacto) / len(impactos))
semianchura_impacto_95 = 1.96 * error_tipico_impacto

print(f"Objetivo: {objetivo:.2f} m")
print(f"Tolerancia: ±{tolerancia:.2f} m")
print(f"Probabilidad estimada de impacto: {probabilidad_impacto:.4f}")
print(f"IC 95% aproximado: [{probabilidad_impacto - semianchura_impacto_95:.4f}, {probabilidad_impacto + semianchura_impacto_95:.4f}]")

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(alcances, bins=40, density=True, alpha=0.75)
plt.axvline(objetivo, linewidth=2, label="Objetivo")
plt.axvline(objetivo - tolerancia, linestyle="--", linewidth=2, label="Zona de impacto")
plt.axvline(objetivo + tolerancia, linestyle="--", linewidth=2)
plt.xlabel("Alcance R (m)")
plt.ylabel("Densidad estimada")
plt.title("Zona objetivo y distribución de alcances")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Convergencia Monte Carlo

Para observar la convergencia del estimador, calculamos la media acumulada

$$
\widehat\mu_n=\frac{1}{n}\sum_{i=1}^n R_i,
$$

y la comparamos con la estimación final obtenida con todas las simulaciones.

In [ ]:
numero_acumulado = np.arange(1, len(alcances) + 1)
media_acumulada = np.cumsum(alcances) / numero_acumulado

plt.figure(figsize=(9, 5))
plt.plot(numero_acumulado, media_acumulada, label="Media acumulada")
plt.axhline(resumen["media"], linestyle="--", label="Media final")
plt.xlabel("Número de simulaciones")
plt.ylabel("Alcance medio estimado (m)")
plt.title("Convergencia del estimador Monte Carlo")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Criterio de parada por tolerancia

Un criterio práctico consiste en detener la simulación cuando la semianchura del intervalo de confianza sea menor que una tolerancia prefijada:

$$
1.96\frac{s_n}{\sqrt n}\leq \varepsilon.
$$

La siguiente función simula por lotes hasta alcanzar la tolerancia deseada o hasta llegar a un número máximo de simulaciones.

In [ ]:
def montecarlo_con_parada(tolerancia_error, tamano_lote=500, maximo_simulaciones=50000, semilla=2026):
    """
    Ejecuta simulaciones Monte Carlo por lotes hasta que la semianchura
    del IC 95% del alcance medio sea menor que tolerancia_error.
    """
    generador_local = np.random.default_rng(semilla)
    alcances_acumulados = []
    historial = []

    total = 0
    while total < maximo_simulaciones:
        nuevos_alcances, *_ = simular_alcances(tamano_lote, generador_local, gravedad)
        alcances_acumulados.extend(nuevos_alcances.tolist())
        total = len(alcances_acumulados)

        if total >= 2:
            resumen_actual = resumen_montecarlo(np.array(alcances_acumulados))
            historial.append((total, resumen_actual["media"], resumen_actual["semianchura_95"]))

            if resumen_actual["semianchura_95"] <= tolerancia_error:
                break

    return np.array(alcances_acumulados), np.array(historial)


tolerancia_error = 10.0  # metros
alcances_parada, historial_parada = montecarlo_con_parada(tolerancia_error=tolerancia_error)

print(f"Número final de simulaciones: {len(alcances_parada)}")
print(f"Media final: {np.mean(alcances_parada):.3f} m")
print(f"Semianchura final del IC 95%: {historial_parada[-1, 2]:.3f} m")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(historial_parada[:, 0], historial_parada[:, 2], marker="o", label="Semianchura IC 95%")
plt.axhline(tolerancia_error, linestyle="--", label="Tolerancia")
plt.xlabel("Número de simulaciones")
plt.ylabel("Semianchura del IC 95% (m)")
plt.title("Criterio de parada por tolerancia")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. Análisis elemental de sensibilidad

Una forma sencilla de estudiar qué parámetros influyen más en el alcance consiste en calcular correlaciones muestrales entre cada parámetro de entrada y el alcance. Este análisis no sustituye a un estudio de sensibilidad más sofisticado, pero resulta útil desde el punto de vista docente.

In [ ]:
def correlacion_muestral(x, y):
    """Calcula el coeficiente de correlación de Pearson."""
    return np.corrcoef(x, y)[0, 1]

correlaciones = {
    "velocidad_inicial": correlacion_muestral(velocidades, alcances),
    "angulo_lanzamiento": correlacion_muestral(np.rad2deg(angulos), alcances),
    "coeficiente_rozamiento": correlacion_muestral(coeficientes, alcances),
    "masa": correlacion_muestral(masas, alcances),
}

for parametro, correlacion in correlaciones.items():
    print(f"Correlación alcance - {parametro}: {correlacion:.4f}")

In [ ]:
parametros = list(correlaciones.keys())
valores_correlacion = [correlaciones[p] for p in parametros]

plt.figure(figsize=(9, 5))
plt.bar(parametros, valores_correlacion)
plt.axhline(0, linewidth=1)
plt.ylabel("Correlación de Pearson")
plt.title("Sensibilidad elemental del alcance respecto de los parámetros")
plt.xticks(rotation=25)
plt.grid(True, axis="y", alpha=0.3)
plt.show()

## 12. Conclusión del experimento computacional

Este notebook muestra cómo un modelo determinista sencillo puede convertirse en un modelo probabilístico al introducir incertidumbre en sus parámetros. El método de Monte Carlo permite estimar:

- el alcance medio;
- la dispersión del alcance;
- intervalos de confianza;
- probabilidades de impacto;
- sensibilidad elemental respecto de los parámetros físicos.

La limitación principal es la tasa de convergencia $N^{-1/2}$, por lo que obtener una reducción importante del error exige aumentar notablemente el número de simulaciones. No obstante, la sencillez del método y su interpretación estadística lo convierten en una herramienta muy adecuada para una primera aplicación física dentro del TFG.